In [3]:
# =========================
# CELL 1: LOAD SORTING + GOOD UNITS
# =========================

import spikeinterface.extractors as se
import pandas as pd
import numpy as np
import os

# ---- PATH ----
phy_path = r"C:\Users\Shermanlab\Desktop\2026-03-05_09-27-03\Record Node 101\experiment2\recording1\continuous\Neuropix-PXI-100.ProbeA\kilosort4"

# ---- LOAD SORTING ----
sorting = se.read_phy(phy_path)
sf = sorting.get_sampling_frequency()

# ---- LOAD PHY LABELS ----
df = pd.read_csv(os.path.join(phy_path, "cluster_group.tsv"), sep="\t")

# ---- GOOD UNITS ----
good_units = df[df["group"] == "good"]["cluster_id"].values.tolist()

print(f"✅ Loaded sorting")
print(f"✅ Good units: {len(good_units)}")

✅ Loaded sorting
✅ Good units: 163


In [4]:
# =========================
# CELL 2: LOAD RECORDING
# =========================

recording_path = r"C:\Users\Shermanlab\Desktop\2026-03-05_09-27-03\Record Node 101\experiment2\recording1"

recording = se.read_openephys(recording_path)

print("✅ Recording loaded")
print(recording)

✅ Recording loaded
OpenEphysBinaryRecordingExtractor: 384 channels - 30.0kHz - 1 segments - 111,882,129 samples 
                                   3,729.40s (1.04 hours) - int16 dtype - 80.02 GiB


In [5]:
# =========================
# CELL 3: LOAD TTL + SELECT PIN
# =========================

ni_folder = r"C:\Users\Shermanlab\Desktop\2026-03-05_09-27-03\Record Node 106\experiment2\recording1\events\NI-DAQmx-105.PXIe-6341\TTL"

TTL_PIN = 7   # 👈 CHANGE THIS

timestamps = np.load(os.path.join(ni_folder, "timestamps.npy"))
full_words = np.load(os.path.join(ni_folder, "full_words.npy"))

# ---- EXTRACT PIN ----
bit_mask = 1 << TTL_PIN
ttl_signal = (full_words & bit_mask) > 0

# ---- RISING EDGES ----
edges = np.where(np.diff(ttl_signal.astype(int)) == 1)[0] + 1
ttl_times = timestamps[edges] - 5.2508   # keep your offset

print(f"⏱️ Found {len(ttl_times)} TTLs on pin {TTL_PIN}")

# ---- PREVIEW ----
print("\nIndex | Time (s)")
for i in range(min(20, len(ttl_times))):
    print(f"{i:5d} | {ttl_times[i]:8.3f}")

⏱️ Found 87 TTLs on pin 7

Index | Time (s)
    0 |  388.110
    1 |  707.522
    2 |  731.483
    3 |  764.085
    4 |  786.375
    5 |  810.686
    6 |  834.253
    7 |  933.558
    8 |  984.970
    9 | 1029.267
   10 | 1060.716
   11 | 1095.766
   12 | 1124.650
   13 | 1151.203
   14 | 1176.921
   15 | 1211.408
   16 | 1241.037
   17 | 1270.941
   18 | 1297.012
   19 | 1329.867


In [38]:
# =========================
# CELL 4: SELECT UNIT + TTL
# =========================

UNIT_ID = 10      # 👈 EXACT PHY UNIT ID
TTL_INDEX = 11     # 👈 WHICH TTL EVENT

# ---- CHECKS ----
if UNIT_ID not in sorting.get_unit_ids():
    raise ValueError("Unit not found in sorting")

if UNIT_ID not in good_units:
    print("⚠️ Not labeled 'good'")
else:
    print("✅ Good unit")

ttl_time = ttl_times[TTL_INDEX]

print(f"\n🎯 Unit: {UNIT_ID}")
print(f"🎯 TTL index: {TTL_INDEX}")
print(f"🎯 TTL time: {ttl_time:.3f} s")

✅ Good unit

🎯 Unit: 10
🎯 TTL index: 18
🎯 TTL time: 1297.017 s


⚠️ centers not found — reconstructing from bin_size


NameError: name 'rate_matrix' is not defined